In [1]:
import os

import scyjava
import imagej
import numpy as np
import xarray
import itertools

from omero import gateway
from getpass import getpass
from pprint import pprint

In [2]:
fiji_path = r"C:\Users\njg135\fiji-MetroloJ\Fiji"
omero_hostname = r"fms-fac-omero.ncl.ac.uk"
omero_username = "njg135"
temp_password = getpass("OMERO Password: ")

In [3]:
ij = imagej.init(fiji_path, mode="interactive")

Double = scyjava.jimport("java.lang.Double")

WindowManager = scyjava.jimport("ij.WindowManager")

MetroloJDialog = scyjava.jimport("metroloJ_QC.setup.MetroloJDialog")
QC_Options = scyjava.jimport("metroloJ_QC.setup.QC_Options")
simpleMetaData = scyjava.jimport("metroloJ_QC.importer.simpleMetaData")

coAlignement = scyjava.jimport("metroloJ_QC.coalignement.coAlignement")
coAlignementReport = scyjava.jimport("metroloJ_QC.report.coAlignementReport")

driftProfiler = scyjava.jimport("metroloJ_QC.stage.driftProfiler")
driftProfilerReport = scyjava.jimport("metroloJ_QC.report.driftProfilerReport")

PSFprofiler = scyjava.jimport("metroloJ_QC.resolution.PSFprofiler")
PSFprofilerReport = scyjava.jimport("metroloJ_QC.report.PSFprofilerReport")

In [ ]:
def initialize_MetroloJDialog(method,
							  image,
							  thresholding_method="Otsu",
							  center_dectection_method="centroid",
							  save_pdf=False,
							  save_csv=False,
							  save_images=False):
	if method == "psf":
		method_string = "PSF profiler report generator"
	elif method == "drift":
		method_string = "Stage positioning and drift report generator"
	elif method == "registration":
		method_string = "Co-registration report generator"
	else:
		raise ValueError("Method must be one of 'psf', 'drift' or 'registration'")

	if thresholding_method not in ["Legacy", "Li", "Minimum", "Otsu"]:
		raise ValueError("Thresholding method must be one of 'Legacy', 'Li', 'Minimum', or 'Otsu'")
	
	if center_dectection_method == "ellipses":
		center_integer = 0
	elif center_dectection_method == "centroid":
		center_integer = 1
	elif center_dectection_method == "max":
		center_integer = 2
	else:
		raise ValueError("Center detection method must be one of 'ellipses', 'centroid', or 'max'")

	WindowManager.setTempCurrentImage(image)
	Dialog = MetroloJDialog(method_string, QC_Options())
	Dialog.beadDetectionThreshold = thresholding_method
	Dialog.centerDetectionMethodIndex = center_integer

	Dialog.savePdf = save_pdf
	Dialog.saveSpreadsheet = save_csv
	Dialog.saveImages = save_images

	return Dialog

def execute_MetroloJ_process(Dialog, report_dir, report_name):
	image = Dialog.ip
	image_title = image.getTitle()
	creationInfo = simpleMetaData.getOMECreationInfos(image, Dialog.debugMode)
	coords = [Double.NaN, Double.NaN]
	
	if Dialog.reportType == "pp":
		execution_instance = PSFprofiler(image, Dialog, image_title, coords, creationInfo)
		report_instance = PSFprofilerReport(image, Dialog, image_title, coords, creationInfo)
	elif Dialog.reportType == "pos":
		execution_instance = driftProfiler(image, Dialog, image_title, coords, creationInfo)
		report_instance = driftProfilerReport(image, Dialog, image_title, coords, creationInfo)
	elif Dialog.reportType == "coa":
		execution_instance = coAlignement(image, Dialog, image_title, coords, creationInfo)
		report_instance = coAlignementReport(image, Dialog, image_title, coords, creationInfo)
	else:
		raise ValueError("Report types supported are PSF profiler, stage positioning and drift and co-registration")
	
	report_instance.saveReport(report_dir, report_name, None)
	
def connect(hostname, username, password):
    """
    Connect to an OMERO server
    :param hostname: Host name
    :param username: User
    :param password: Password
    :return: Connected BlitzGateway
    """
    conn = gateway.BlitzGateway(username, password,
                        host=hostname, secure=True, port=4063)
    conn.connect()
    conn.c.enableKeepAlive(60)
    return conn


def disconnect(conn):
    """
    Disconnect from an OMERO server
    :param conn: The BlitzGateway
    """
    conn.close()

class ChannelObject:
	def __init__(self, channel):
		self.channel = channel
		self.name = channel.getName()
		self.emission_wave = channel.getEmissionWave()
		self.excitation_wave = channel.getExcitationWave()
		self.mode = channel.getLogicalChannel().getMode().value
    
class ImageObject:
	def __init__(self, image):
		self.image = image
		self.id = image.getId()
		self.name = image.getName()
		self.size_x = image.getSizeX()
		self.size_y = image.getSizeY()
		self.size_z = image.getSizeZ()
		self.size_c = image.getSizeC()
		self.size_t = image.getSizeT()
		self.pixels = image.getPrimaryPixels()
		self.scale_x = self.pixels.getPhysicalSizeX()
		self.scale_y = self.pixels.getPhysicalSizeY()
		self.scale_z = self.pixels.getPhysicalSizeZ()
		self.dim_order = "TCZYX"
		self.objective = image.getObjectiveSettings()
		self.refractive_index = self.objective.getRefractiveIndex()
		self.NA = self.objective.getObjective().getLensNA()
		self.channels = [ChannelObject(ch) for ch in image.getChannels()]
		self.shape = (self.size_t, self.size_c, self.size_z, self.size_y, self.size_x)
		self.image_data = None

	def load_plane(self, c, t, z):
		self.image_data[t, c, z, :, :] = np.array(self.pixels.getPlane(z, c, t))

	def load_image_data(self, c=None, t=None, z=None):
		if c is None:
			c = list(range(self.size_c))
		if t is None:
			t = list(range(self.size_t))
		if z is None:
			z = list(range(self.size_z))

		self.image_data = np.zeros((len(t), len(c), len(z), self.size_y, self.size_x))
		all_iterations = list(itertools.product(c, t, z))
		for args in all_iterations:
			self.load_plane(*args)
		self.shape = self.image_data.shape
	
	def generate_ImagePlus(self):
		if self.image_data is None:
			self.load_image_data()
		image_plus = ij.py.to_java(self.image_data.values)
		image_plus.setCalibration(self.scale_x.getValue(), self.scale_y.getValue(), self.scale_z.getValue())
		return image_plus

In [9]:
connection = gateway.BlitzGateway(omero_username, temp_password, host=omero_hostname)
print ("Connecting to OMERO server...")
connection.connect()
print ("Connection successful!")

Connecting to OMERO server...
Connection successful!


In [18]:
confocal_obj = ImageObject(connection.getObject("Image", 136561))
widefield_obj = ImageObject(connection.getObject("Image", 130658))

In [22]:
confocal_obj.scale_x.getUnit()

MICROMETER

In [14]:
confocal_obj.load_image_data()

In [12]:
widefield_obj.load_image_data()

In [81]:
print (confocal_obj.channels[0].mode)
print (widefield_obj.channels[0].mode)

LaserScanningConfocalMicroscopy
WideField


In [71]:
image_obj.channels[0].channel.getLogicalChannel().getMode().value

'WideField'

In [7]:
objsettings = img.getObjectiveSettings()
objective = objsettings.getObjective()
NA = objective.getLensNA()